# Assignment Testing

This notebook will:

- Generate a dataset of X Random User Ids
- Create a number of experiments with set buckets assigned
- Fetch assignments for each user id from the assignment service
- Verify each experiment achieved a statisitcally even distribution between assignments


## Requirements
- The following services must be running:
    - Assignment Service (http://localhost:8082)
    - Experimentation Service
    - Postgres
- You **MUST** run the `experiments.sql` script to create the experiments and buckets in the database before running this notebook



In [7]:
NUM_USERS = 1_000_000

## Generate Random User Ids

In [8]:
import uuid

import pandas as pd
import warnings
# One of the confidence usages of pandas triggers future warning which are a pain to see.
warnings.filterwarnings("ignore", category=FutureWarning)


user_ids = [str(uuid.uuid4()) for _ in range(NUM_USERS)]

## Make API Calls to assignment service

In [9]:
import asyncio
import aiohttp

CONCURRENCY_LIMIT = 1000

async def fetch(session, semaphore, user_id):
    async with semaphore:
        async with session.get(f"http://localhost:8082/api/assignments/{user_id}") as resp:
            return await resp.json()

async def main(user_ids):
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, semaphore, user_id) for user_id in user_ids]
        return await asyncio.gather(*tasks)
    return results

results = await main(user_ids)

print(f"Total results: {len(results)}")
if len(results) != NUM_USERS:
    print(f"Warning: Expected {NUM_USERS} results, but got {len(results)}")



Total results: 1000000


## Transform results into a dataframe

In [10]:
assignments = {}


for result in results:
    if not result:
        continue
    for feature_flag, variant_id  in result.items():
        if feature_flag not in assignments:
            assignments[feature_flag] = {
                variant_id: 1
            }
        else:
            if variant_id not in assignments[feature_flag]:
                assignments[feature_flag][variant_id] = 1
            else:
                assignments[feature_flag][variant_id] += 1


records = []
for experiment, variants in assignments.items():
  for variant, count in variants.items():
      records.append({
          "experiment": experiment,
          "variant": variant,
          "count": count
      })

df = pd.DataFrame(records)
df["total"] = df.groupby("experiment")["count"].transform("sum")



## Perform Chi-Squared Test

In [11]:
import spotify_confidence as confidence

test = confidence.ChiSquared(
    data_frame=df,
    numerator_column="count",
    denominator_column="total",
    categorical_group_columns=["experiment", "variant"]
)

pairs = [
    (("feature_flag_1", "flag_1_variant_1"), ("feature_flag_1", "flag_1_variant_2")),
    (("feature_flag_2", "flag_2_variant_1"), ("feature_flag_2", "flag_2_variant_2")),
    (("feature_flag_3", "flag_3_variant_1"), ("feature_flag_3", "flag_3_variant_2")),
    (("feature_flag_4", "flag_4_variant_1"), ("feature_flag_4", "flag_4_variant_2")),
    (("feature_flag_5", "flag_5_variant_1"), ("feature_flag_5", "flag_5_variant_2")),
]

results = []
for level_1, level_2 in pairs:
    result = test.difference(level_1=level_1, level_2=level_2)
    results.append(result)

results_df = pd.concat(results)




## Apply Bonferroni Correction


In [12]:



results_df["adjusted_p"] = results_df["p-value"] * len(results_df)
results_df["afterCorrection"] = results_df["adjusted_p"] < 0.05


results_df



,level_1,level_2,absolute_difference,difference,ci_lower,ci_upper,p-value,adjusted ci_lower,adjusted ci_upper,adjusted p-value,is_significant,powered_effect,required_sample_size,adjusted_p,afterCorrection
0,"(feature_flag_1, flag_1_variant_1)","(feature_flag_1, flag_1_variant_2)",True,0.002605,-0.001782,0.006992,0.244471,-0.001782,0.006992,0.244471,False,None,None,1.222353,False
0,"(feature_flag_2, flag_2_variant_1)","(feature_flag_2, flag_2_variant_2)",True,-0.001775,-0.006163,0.002614,0.428018,-0.006163,0.002614,0.428018,False,None,None,2.140092,False
0,"(feature_flag_3, flag_3_variant_1)","(feature_flag_3, flag_3_variant_2)",True,-0.001855,-0.006231,0.002522,0.406173,-0.006231,0.002522,0.406173,False,None,None,2.030866,False
0,"(feature_flag_4, flag_4_variant_1)","(feature_flag_4, flag_4_variant_2)",True,-0.001829,-0.006211,0.002552,0.413224,-0.006211,0.002552,0.413224,False,None,None,2.066120,False
0,"(feature_flag_5, flag_5_variant_1)","(feature_flag_5, flag_5_variant_2)",True,0.002033,-0.002353,0.006419,0.363603,-0.002353,0.006419,0.363603,False,None,None,1.818016,False
